In [ ]:
# rende config.py (in notebooks/) importabile anche dai notebook in sottocartelle
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from scipy import stats

import matplotlib.patches as mpatches

import concurrent.futures
from concurrent.futures import ProcessPoolExecutor

import logging

import albedo_functions as af

In [ ]:
# Configurazione logging
log_file = "process_log.txt"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file, mode='w'),  # Write to file
    ]
)

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
var='tas'
era_var='2t'
COV_PATH = f'{POST_DATA}/'
LAI_PATH = f'{CONFESS_DATA}/'
SAVE_PATH = f'{WORK_DIR}/'
    
leads = [0, 1, 2, 3, 4]

In [ ]:
dset_ctrl = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_ctrl_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_land_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_land_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_ctrl_ocean = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_ocean_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens_ocean = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_ocean_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)

dset_ctrl_em = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_em_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens_em = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_em_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_ctrl_em_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_em_land_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens_em_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_em_land_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_ctrl_em_ocean= xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_em_ocean_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens_em_ocean = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_em_ocean_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)

ld = xr.DataArray(leads, dims='lead_time', name='lead_time')
dset_ctrl['lead_time']=ld
dset_sens['lead_time']=ld
dset_ctrl_em['lead_time']=ld
dset_sens_em['lead_time']=ld

dset_ctrl_land['lead_time']=ld
dset_sens_land['lead_time']=ld
dset_ctrl_em_land['lead_time']=ld
dset_sens_em_land['lead_time']=ld


dset_ctrl_ocean['lead_time']=ld
dset_sens_ocean['lead_time']=ld
dset_ctrl_em_ocean['lead_time']=ld
dset_sens_em_ocean['lead_time']=ld

# Convert the dataset to a DataFrame
dset_ctrl = dset_ctrl.to_dataframe().reset_index()
dset_sens = dset_sens.to_dataframe().reset_index()
dset_ctrl_em = dset_ctrl_em.to_dataframe().reset_index()
dset_sens_em = dset_sens_em.to_dataframe().reset_index()

dset_ctrl_land = dset_ctrl_land.to_dataframe().reset_index()
dset_sens_land = dset_sens_land.to_dataframe().reset_index()
dset_ctrl_em_land = dset_ctrl_em_land.to_dataframe().reset_index()
dset_sens_em_land = dset_sens_em_land.to_dataframe().reset_index()

dset_ctrl_ocean = dset_ctrl_ocean.to_dataframe().reset_index()
dset_sens_ocean = dset_sens_ocean.to_dataframe().reset_index()
dset_ctrl_em_ocean = dset_ctrl_em_ocean.to_dataframe().reset_index()
dset_sens_em_ocean = dset_sens_em_ocean.to_dataframe().reset_index()

# --- limiti y comuni: unione dei dati dei tre pannelli (margine 10%) ---
_dfs = [dset_ctrl, dset_sens, dset_ctrl_land, dset_sens_land, dset_ctrl_ocean, dset_sens_ocean]
_ymin = min(d[var].min() for d in _dfs)
_ymax = max(d[var].max() for d in _dfs)
_margin = 0.10 * (_ymax - _ymin)
YLIM = (_ymin - _margin, _ymax + _margin)

# --- significativita' bootstrap (calcolata nel notebook di calcolo, letta qui) ---
def _load_sig(region_suffix):
    import glob
    pattern = f'{SAVE_PATH}/{exp_sens}-{exp_ctrl}_{var}_lead_*_1x1_global_accsig{region_suffix}_1999.nc'
    files = sorted(glob.glob(pattern))
    if not files:
        print(f"[significativita] file mancanti per {region_suffix or 'global'!r}: "
              f"esegui prima 05-global_acc.ipynb per generarli. Asterischi saltati.")
        return [0] * len(leads)
    ds = xr.open_mfdataset(files, concat_dim='lead_time', combine='nested', chunks=-1)
    return ds['sig'].values

sig_global = _load_sig('')
sig_land = _load_sig('_land')
sig_ocean = _load_sig('_ocean')

def add_significance(ax, sig):
    """Segna con '*' i lead significativi (bootstrap, dal notebook di calcolo)."""
    y0, y1 = ax.get_ylim()
    ypos = y1 - 0.05 * (y1 - y0)
    for xi in range(len(leads)):
        if sig[xi]:
            ax.text(xi, ypos, '*', ha='center', va='top', fontsize=20, fontweight='bold')


In [ ]:
# stile hue coerente con la Fig.3
dset_sens['exp'] = 'SENS'
dset_ctrl['exp'] = 'CTRL'
df = pd.concat([dset_sens, dset_ctrl])

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(x='lead_time', y='tas', hue='exp', hue_order=['SENS', 'CTRL'],
            data=df, palette={'SENS': 'lightcoral', 'CTRL': 'lavender'}, ax=ax)

# medie d'ensemble centrate sul box corrispondente (2 hue, width 0.8 -> +/- 0.2)
off = 0.2
x = np.arange(len(leads))
ax.plot(x - off, dset_sens_em.sort_values('lead_time')[var].values, color='k')                  # SENS
ax.plot(x + off, dset_ctrl_em.sort_values('lead_time')[var].values, color='k', linestyle='--')  # CTRL
ax.set_ylim(*YLIM)
ax.set_ylabel('ACC')
add_significance(ax, sig_global)
ax.set_title('a) GLOBAL')

fig.savefig(f'{FIG_DIR}/{var}_acc_global_1999.png', dpi=600, bbox_inches='tight')


In [ ]:
# stile hue coerente con la Fig.3
dset_sens_land['exp'] = 'SENS'
dset_ctrl_land['exp'] = 'CTRL'
df = pd.concat([dset_sens_land, dset_ctrl_land])

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(x='lead_time', y='tas', hue='exp', hue_order=['SENS', 'CTRL'],
            data=df, palette={'SENS': 'lightcoral', 'CTRL': 'lavender'}, ax=ax)

# medie d'ensemble centrate sul box corrispondente (2 hue, width 0.8 -> +/- 0.2)
off = 0.2
x = np.arange(len(leads))
ax.plot(x - off, dset_sens_em_land.sort_values('lead_time')[var].values, color='k')                  # SENS
ax.plot(x + off, dset_ctrl_em_land.sort_values('lead_time')[var].values, color='k', linestyle='--')  # CTRL
ax.set_ylim(*YLIM)
ax.set_ylabel('ACC')
add_significance(ax, sig_land)
ax.set_title('b) LAND ONLY')

fig.savefig(f'{FIG_DIR}/{var}_acc_land_1999.png', dpi=600, bbox_inches='tight')


In [ ]:
# stile hue coerente con la Fig.3
dset_sens_ocean['exp'] = 'SENS'
dset_ctrl_ocean['exp'] = 'CTRL'
df = pd.concat([dset_sens_ocean, dset_ctrl_ocean])

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(x='lead_time', y='tas', hue='exp', hue_order=['SENS', 'CTRL'],
            data=df, palette={'SENS': 'lightcoral', 'CTRL': 'lavender'}, ax=ax)

# medie d'ensemble centrate sul box corrispondente (2 hue, width 0.8 -> +/- 0.2)
off = 0.2
x = np.arange(len(leads))
ax.plot(x - off, dset_sens_em_ocean.sort_values('lead_time')[var].values, color='k')                  # SENS
ax.plot(x + off, dset_ctrl_em_ocean.sort_values('lead_time')[var].values, color='k', linestyle='--')  # CTRL
ax.set_ylim(*YLIM)
ax.set_ylabel('ACC')
add_significance(ax, sig_ocean)
ax.set_title('c) OCEAN ONLY')

fig.savefig(f'{FIG_DIR}/{var}_acc_ocean_1999.png', dpi=600, bbox_inches='tight')


In [ ]:
# composito 1x3: unisce global / land / ocean in un'unica figura (come Fig.3)
from matplotlib.gridspec import GridSpec
import matplotlib.image as mpimg

img1 = mpimg.imread(f'{FIG_DIR}/{var}_acc_global_1999.png')
img2 = mpimg.imread(f'{FIG_DIR}/{var}_acc_land_1999.png')
img3 = mpimg.imread(f'{FIG_DIR}/{var}_acc_ocean_1999.png')

fig = plt.figure(figsize=(10, 8), constrained_layout=False)
gs = GridSpec(1, 3, wspace=0.0, hspace=0.0)
axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
for ax, img in zip(axes, [img1, img2, img3]):
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout(pad=0.0)
plt.savefig(f'{FIG_DIR}/fig5_{var}_acc_1999.png', dpi=600, bbox_inches='tight')


In [ ]:
# ============================================================================
# ACC per COMBINAZIONI di lead year: 5 singoli + 10 range = 15 punti sull'asse x.
# Legge i file dei lead singoli (lead_{n}) gia' esistenti e quelli dei range
# (leadyears_{y1}-{y2}) prodotti dal calcolo 05-global_acc. Salta i mancanti.
# ============================================================================
import glob
import itertools

def _win(lab):
    # lunghezza della finestra di media: 1 per i singoli, y2-y1+1 per i range
    if '-' in lab:
        a, b = lab.split('-'); return int(b) - int(a) + 1
    return 1

LABELS = ['0', '1', '2', '3', '4',
          '0-1', '1-2', '2-3', '3-4',
          '0-2', '1-3', '2-4',
          '0-3', '1-4',
          '0-4']

def _acc_file(exp, label, suff, kind=''):
    base = f"leadyears_{label}" if '-' in label else f"lead_{label}"
    return f"{SAVE_PATH}/{exp}_{var}_{base}_1x1_global_acc{kind}{suff}_1999.nc"

def _sig_file(label, suff):
    base = f"leadyears_{label}" if '-' in label else f"lead_{label}"
    return f"{SAVE_PATH}/{exp_sens}-{exp_ctrl}_{var}_{base}_1x1_global_accsig{suff}_1999.nc"

def load_region_all(suff):
    rows, em_s, em_c, sig, used = [], {}, {}, {}, []
    for lab in LABELS:
        fs, fc = _acc_file(exp_sens, lab, suff), _acc_file(exp_ctrl, lab, suff)
        if not (glob.glob(fs) and glob.glob(fc)):
            print(f"[lead years] manca {lab}{suff or ' global'}, salto")
            continue
        used.append(lab)
        vs = np.ravel(xr.open_dataset(fs)[var].values); vs = vs[~np.isnan(vs)]
        vc = np.ravel(xr.open_dataset(fc)[var].values); vc = vc[~np.isnan(vc)]
        rows += [(lab, float(v), 'SENS') for v in vs]
        rows += [(lab, float(v), 'CTRL') for v in vc]
        em_s[lab] = float(xr.open_dataset(_acc_file(exp_sens, lab, suff, '_em'))[var].values)
        em_c[lab] = float(xr.open_dataset(_acc_file(exp_ctrl, lab, suff, '_em'))[var].values)
        sig[lab] = float(xr.open_dataset(_sig_file(lab, suff))['sig'].values) if glob.glob(_sig_file(lab, suff)) else 0.0
    return pd.DataFrame(rows, columns=['lead_label', 'tas', 'exp']), used, em_s, em_c, sig

_regions = {'': 'a) GLOBAL', '_land': 'b) LAND ONLY', '_ocean': 'c) OCEAN ONLY'}
_data = {suff: load_region_all(suff) for suff in _regions}
_allv = np.concatenate([d[0]['tas'].values for d in _data.values() if len(d[0])])
_m = 0.10 * (_allv.max() - _allv.min())
YLIM_ALL = (_allv.min() - _m, _allv.max() + _m)

def plot_panel_all(suff, title, fname):
    df, used, em_s, em_c, sig = _data[suff]
    fig, ax = plt.subplots(figsize=(22, 6))
    sns.boxplot(x='lead_label', y='tas', hue='exp', hue_order=['SENS', 'CTRL'], order=used,
                data=df, palette={'SENS': 'lightcoral', 'CTRL': 'lavender'}, ax=ax)
    off = 0.2
    # linea della media d'ensemble spezzata tra i gruppi (per lunghezza di finestra)
    for _w, _grp in itertools.groupby(range(len(used)), key=lambda i: _win(used[i])):
        idxs = np.array(list(_grp))
        ax.plot(idxs - off, [em_s[used[i]] for i in idxs], color='k')                  # SENS
        ax.plot(idxs + off, [em_c[used[i]] for i in idxs], color='k', linestyle='--')  # CTRL
    ax.set_ylim(*YLIM_ALL)
    ax.set_ylabel('ACC'); ax.set_xlabel('lead years')
    y0, y1 = ax.get_ylim(); ypos = y1 - 0.05 * (y1 - y0)
    for xi, l in enumerate(used):
        if sig.get(l):
            ax.text(xi, ypos, '*', ha='center', va='top', fontsize=18, fontweight='bold')
    ax.set_title(title)
    plt.setp(ax.get_xticklabels(), rotation=45)
    fig.savefig(f'{FIG_DIR}/{fname}', dpi=600, bbox_inches='tight')

plot_panel_all('', 'a) GLOBAL', f'{var}_acc_leadyears_global_1999.png')
plot_panel_all('_land', 'b) LAND ONLY', f'{var}_acc_leadyears_land_1999.png')
plot_panel_all('_ocean', 'c) OCEAN ONLY', f'{var}_acc_leadyears_ocean_1999.png')


In [ ]:
# composito verticale (3x1) delle combinazioni di lead year
from matplotlib.gridspec import GridSpec
import matplotlib.image as mpimg

imgs = [mpimg.imread(f'{FIG_DIR}/{var}_acc_leadyears_{r}_1999.png') for r in ['global', 'land', 'ocean']]
fig = plt.figure(figsize=(14, 16))
gs = GridSpec(3, 1, hspace=0.05)
for i, img in enumerate(imgs):
    ax = fig.add_subplot(gs[i, 0]); ax.imshow(img); ax.axis('off')
plt.tight_layout(pad=0.2)
plt.savefig(f'{FIG_DIR}/fig5b_{var}_acc_leadyears_1999.png', dpi=600, bbox_inches='tight')
